# QC-large mesh visualization, part 1

This notebook shows the first half of the representative observed-pair cases.

Each case has two visual blocks:

- **Solid mesh comparison:** the baseline/source ground-truth mesh, the observed future/target
  ground-truth mesh, and one predicted future mesh from each model. These meshes are rendered as
  opaque solids. A display-only repair note appears when the viewer filled small visual holes or
  fixed mesh bookkeeping; the numerical metrics still come from the saved evaluation outputs.
- **Surface-change map:** each surface is rigidly aligned to the source mesh. Color shows how far
  each displayed future surface moved away from the source surface in millimeters. This is a
  change-from-baseline map, not a direct error heatmap against the target.


In [1]:
from pathlib import Path
import importlib
import sys

import pandas as pd
from IPython.display import Markdown, display

root = Path.cwd().resolve()
while not (root / ".git").exists():
    if root.parent == root:
        raise RuntimeError("Could not locate repo root from current working directory.")
    root = root.parent

script_dir = root / "examples" / "ADNI_1_L_No_MCI" / "brainode_comparison_task3_core_brainode_original" / "scripts"
if str(script_dir) not in sys.path:
    sys.path.insert(0, str(script_dir))

import longitudinal_visual_notebook_support as lv
lv = importlib.reload(lv)

ctx = lv.create_context(device="auto")
display(Markdown(
    "Using `longitudinal_visual_notebook_support.py`. "
    "Observed-pair plots are light. Anchor forecast cells decode meshes for the SIREN models and run best on CUDA."
))


Using `longitudinal_visual_notebook_support.py`. Observed-pair plots are light. Anchor forecast cells decode meshes for the SIREN models and run best on CUDA.

In [2]:
cases = ctx.representative_cases()
first_half = cases[:3]
pd.DataFrame(
    [{"split": c.split, "diagnosis": c.diagnosis, "subject_id": c.subject_id} for c in first_half]
)


,split,diagnosis,subject_id
0,train,CN,31
1,train,AD,995
2,val,CN,677


In [3]:
for case in first_half:
    display(Markdown(f"## {case.split.upper()} {case.diagnosis} | subject {case.subject_id}"))
    display(Markdown(
        "### Case metadata\n"
        "This table identifies the selected observed pair. The source scan is the input shape and "
        "the target scan is the real later scan used as ground truth for this case."
    ))
    display(ctx.case_overview_table(case))
    display(Markdown(
        "### Solid mesh comparison\n"
        "- **Source GT:** real baseline mesh given to the model.\n"
        "- **Target GT:** real future mesh at the later scan age.\n"
        "- **Model panels:** predicted future meshes from BrainODE PCA, PCA cocycle flow, "
        "SIREN cocycle flow, and SIREN latent ODE.\n"
        "The panel is for visual plausibility and gross anatomical comparison. It is not colored by error."
    ))
    fig = ctx.plot_case_mesh_panel(case)
    fig.show()
    display(Markdown(
        "### Surface-change maps\n"
        "These maps compare each future surface to the same source mesh after rigid alignment. "
        "Darker/lower values mean little surface movement from baseline; brighter/higher values mean "
        "larger local displacement from baseline. The Target GT panel shows the real change, and the "
        "model panels show the predicted change pattern."
    ))
    heat_fig, heat_table = ctx.plot_case_change_heatmaps(case)
    heat_fig.show()
    display(Markdown(
        "The table below summarizes the displayed change map: mean, p95, and max are surface-shift "
        "magnitudes in millimeters. Failed mesh decodes are kept in the table instead of stopping the notebook."
    ))
    display(heat_table)


## TRAIN CN | subject 31

### Case metadata
This table identifies the selected observed pair. The source scan is the input shape and the target scan is the real later scan used as ground truth for this case.

,model,chosen_transport,gap_years,chamfer_l2_squared,assd_mm,hd95_mm,volume_abs_error_cm3,surface_area_abs_error_cm2
0,BrainODE PCA150,endpoint,10.255989,0.002070,0.540511,1.487205,0.559828,1.883015
1,PCA150 cocycle flow,composed,10.255989,0.000857,0.376803,0.988132,0.062958,0.054143
2,SIREN cocycle flow,direct,10.255988,0.002718,0.675967,1.649827,0.535662,5.268731
3,SIREN latent ODE,direct,10.255988,0.003244,0.775844,1.719510,0.515956,1.458915


### Solid mesh comparison
- **Source GT:** real baseline mesh given to the model.
- **Target GT:** real future mesh at the later scan age.
- **Model panels:** predicted future meshes from BrainODE PCA, PCA cocycle flow, SIREN cocycle flow, and SIREN latent ODE.
The panel is for visual plausibility and gross anatomical comparison. It is not colored by error.

### Surface-change maps
These maps compare each future surface to the same source mesh after rigid alignment. Darker/lower values mean little surface movement from baseline; brighter/higher values mean larger local displacement from baseline. The Target GT panel shows the real change, and the model panels show the predicted change pattern.

...

,model,chosen_transport,gap_years,chamfer_l2_squared,assd_mm,hd95_mm,volume_abs_error_cm3,surface_area_abs_error_cm2
0,BrainODE PCA150,endpoint,10.017792,0.000930,0.407103,0.979004,0.011041,0.209210
1,PCA150 cocycle flow,composed,10.017792,0.000868,0.399677,1.014496,0.194782,0.782563
2,SIREN cocycle flow,direct,10.017795,0.001545,0.520756,1.351631,0.342225,5.186007
3,SIREN latent ODE,composed,10.017795,0.001278,0.483083,1.189915,0.475163,2.044264


### Solid mesh comparison
- **Source GT:** real baseline mesh given to the model.
- **Target GT:** real future mesh at the later scan age.
- **Model panels:** predicted future meshes from BrainODE PCA, PCA cocycle flow, SIREN cocycle flow, and SIREN latent ODE.
The panel is for visual plausibility and gross anatomical comparison. It is not colored by error.

### Surface-change maps
These maps compare each future surface to the same source mesh after rigid alignment. Darker/lower values mean little surface movement from baseline; brighter/higher values mean larger local displacement from baseline. The Target GT panel shows the real change, and the model panels show the predicted change pattern.

The table below summarizes the displayed change map: mean, p95, and max are surface-shift magnitudes in millimeters. Failed mesh decodes are kept in the table instead of stopping the notebook.

,label,mean_shift_mm,p95_shift_mm,max_shift_mm,icp_cost,status,error
0,Target GT,0.444937,0.820400,1.339131,0.000178,ok,
1,BrainODE PCA150 (endpoint),0.374254,0.624128,0.760225,0.000108,ok,
2,PCA150 cocycle flow (composed),0.321537,0.501734,0.731241,0.000078,ok,
3,SIREN cocycle flow (direct),0.387633,0.769219,1.204482,0.000120,ok,
4,SIREN latent ODE (composed),0.311334,0.533900,0.941419,0.000069,ok,
